In [251]:
%pip install -U langchain-groq langchain-core langchain-community requests pydantic



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [307]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests
import os

In [308]:
from langchain_core.messages import HumanMessage, BaseMessage,ToolMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated


In [309]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("EXCHANGE_RATE_API_KEY")
print("API key loaded:", bool(api_key))

API key loaded: True


In [310]:
# tool create
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/5954a27cbdd9de0991afc860/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate


In [311]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [312]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791676801,
 'time_last_update_utc': 'Sun, 11 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791763201,
 'time_next_update_utc': 'Mon, 12 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.8297}

In [313]:
convert.invoke({'base_currency_value':10, 'conversion_rate':96.8297})

968.297

In [314]:
llm = ChatGroq(model="openai/gpt-oss-120b")


In [315]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [316]:

import json
from langchain_core.messages import HumanMessage, ToolMessage

def run_with_tools(llm_tool, messages):

    conversion_rate = None

    while True:
        ai_message = llm_tool.invoke(messages)
        messages.append(ai_message)

        print("Tool calls:", ai_message.tool_calls)

        if not ai_message.tool_calls:
            return ai_message.content

        for tool_call in ai_message.tool_calls:
            name = tool_call["name"]
            args = tool_call["args"]

            if name == "get_conversion_factor":
                result = get_conversion_factor.invoke(args)
                conversion_rate = result["conversion_rate"]

            elif name == "convert":
                args["conversion_rate"] = conversion_rate
                result = convert.invoke(args)

            else:
                raise ValueError(f"Unknown tool: {name}")

            messages.append(
                ToolMessage(
                    content=json.dumps(result),
                    tool_call_id=tool_call["id"]
                )
            )


In [317]:
messages = [
    HumanMessage(
        content=(
            "What is the conversion factor between INR and USD, "
            "and convert 10 USD to INR using the convert tool."
        )
    )
]

response = run_with_tools(llm_with_tools, messages)

print("Final answer:", response)

Tool calls: [{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'fc_ca000df5-d91a-42db-b45e-c8450ee53c9d', 'type': 'tool_call'}]
Tool calls: [{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'fc_1d46b6f1-dda3-48dd-bee4-34c58f104252', 'type': 'tool_call'}]
Tool calls: [{'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'fc_bf3dbf1c-0ee0-48b5-a23a-ec83b66be491', 'type': 'tool_call'}]
Tool calls: []
Final answer: The current conversion rates are:

- **1 INR ≈ 0.01033 USD** (i.e., 1 USD ≈ 96.83 INR)  
- **1 USD ≈ 96.83 INR**

Using the USD‑to‑INR rate, **10 USD ≈ 968.30 INR**.


In [301]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and convert 10 USD to INR using the convert tool.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get conversion factor between INR and USD. The user asks: "What is the conversion factor between INR and USD, and convert 10 USD to INR using the convert tool."\n\nWe need to fetch conversion factor: base_currency INR? Or USD? Usually conversion factor between INR and USD: maybe 1 USD = X INR. They want conversion factor between INR and USD. Could interpret as factor from INR to USD? But then converting 10 USD to INR. So we need conversion factor for USD->INR. Let\'s fetch conversion factor with base_currency USD, target_currency INR. Use get_conversion_factor. Then use convert to convert 10 USD to INR. The convert tool expects base_currency_value? Probably also needs conversion factor? The spec: given a currency conversion rate this function c